# 01 — Cleaning & Feature Engineering

Load the raw hand-digitised fuel log, type it, normalise currency to euros,
and build the derived fields. The output is saved to `data/processed/` for
the later notebooks.

**Key caveat:** fill-ups were done to a *fixed amount of money*, not to a full
tank, so per-leg fuel economy is noisy. Consumption is measured in aggregate
(see notebook 04).

In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable from the notebooks folder.
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
from galloper import io, cleaning, features, viz

viz.set_style()
pd.set_option("display.max_columns", None)

## 1. Load raw

In [ ]:
df_raw = io.load_raw()
print(df_raw.shape)
df_raw.head()

## 2. Clean: types + currency normalisation

In [ ]:
df = cleaning.clean(df_raw)
df[["odometer_km", "litres", "amount_eur", "currency", "date",
    "brand", "place", "price_per_l"]].head()

## 3. Trip features and validity flags

In [ ]:
df = features.add_trip_features(df)
df = features.flag_price_outliers(df)

# Quick data-quality overview.
print("rows:", len(df))
print("missing litres:", df["litres"].isna().sum())
print("missing odometer:", df["odometer_km"].isna().sum())
print("missing date:", df["date"].isna().sum())
print("valid legs for economy:", df["leg_valid"].sum())
print("price outliers flagged:", df["price_outlier"].sum())

### Rows flagged as price outliers (candidate transcription errors)

In [ ]:
df.loc[df["price_outlier"],
       ["date", "odometer_km", "litres", "amount_eur",
        "price_per_l", "price_rel_dev", "brand", "place", "notes"]]

## 4. Aggregate views (the correct consumption lens)

In [ ]:
yearly = features.yearly_summary(df)
yearly

In [ ]:
roll = features.rolling_economy(df, window=10)
roll.tail()

## 5. Persist processed data

In [ ]:
io.save_processed(df, "fuel_clean")
io.save_processed(yearly, "yearly_summary")
print("Saved to data/processed/")